In [112]:
include("Optimisation.jl")
verbose = true

true

In [40]:
function f(x::Float64)
    x^2 - 1
end


f (generic function with 1 method)

In [41]:
function fder(x::Float64)
    2 * x
end

fder (generic function with 1 method)

In [42]:
x0 = 10.

10.0

In [43]:
OptimNewton(f,fder, x0)

1.0

In [44]:
function func(x::Vector{Float64})::Float64
    (x[1]-1)^2 + x[2]^2 + x[3]^4
end

function funcder(x::Vector{Float64})::Vector{Float64}
     [ 2 * (x[1]-1) ; 2 * x[2] ; 4 * x[3]^3]
end

# How to measure the distance to zero of the gradient of f
function distance1(x::Vector{Float64})::Float64
    return sqrt(x[1]^2 + x[2]^2 + x[3]^2)
end

# How to evaluate the distance between two consecutive steps
function distance2(x::Vector{Float64})::Float64
    return sqrt(x[1]^2 + x[2]^2 + x[3]^2)
end
#problème

x0 = [0. ; 0. ; 0.]
step = 0.4
OptimGradient(func, funcder, x0, step, distance1, distance2)

Distance between two consecutive updates is small
The final value of the gradient norm (distance 1) is : 5.070610598068015e-11
The final value of the cost is : 6.427772959309918e-22


3-element Vector{Float64}:
 0.999999999974647
 0.0
 0.0

In [45]:
function constrain(x::Vector{Float64})::Float64
    return x[1] - 0.5
end
function derconstrain(x::Vector{Float64})::Vector{Float64}
    return [1.; 0.; 0.]
end
beta = 2.
mu0 = 0.

0.0

In [46]:
#AugmentedLagrangian(func, funcder, constrain,derconstrain, x0, mu0, step, beta, distance1, distance2)

In [47]:
function proj(x::Vector{Float64})::Vector{Float64}
    return max.(0.,x)
end
x0 = [-20., 3., -15.]

3-element Vector{Float64}:
 -20.0
   3.0
 -15.0

In [48]:
#OptimProjectedGradient(func, funcder, proj, x0, step, distance1, distance2)

In [49]:
OptimGradient(func, funcder, x0, step, distance1, distance2, proj)

Distance between two consecutive updates is small
The final value of the gradient norm (distance 1) is : 8.2039690106389e-11
The final value of the cost is : 1.682627688188085e-21


3-element Vector{Float64}:
 0.9999999999870193
 3.891180282460559e-11
 0.0

Levenberg-Marquardt
On prend pour exemple un problème de régression linéaire 
On se donne beaucoup de data et 2 paramètres à optimiser

In [5]:
include("Optimisation.jl")
Ndata = 300
X = [i for i in 1:Ndata]
F = -5. .* X .+ 12. + cos.(X)
param = [15. ; 20.]

function sumres(param::Vector{Float64})::Float64
    s = 0.
    for q in 1:Ndata
        s+= ( param[1] * X[q] + param[2] - F[q])^2 
    end
    return s 
end

function dersumres(param::Vector{Float64})::Vector{Float64}
    # renvoie le gradient de l function coût
    m = size(param)[1]
    s1 = 0.
    s2 = 0.
    for q in 1:Ndata
        s1+=  X[q] * ( param[1] * X[q] + param[2] - F[q])
        s2 +=  ( param[1] * X[q] + param[2] - F[q])
    end
    return 2. * [s1  ; s2 ]
end

function resder(param::Vector{Float64})::Symmetric{Float64, Matrix{Float64}}
    # doit renvoyer une matrice symétrique de taille 2 * 2 ici
    m = size(param)[1]
    s11 = 0.
    s12 = 0.
    s22 = 0.
    for q in 1:Ndata
        s11 +=  X[q]^2 
        s22 += 1.
        s12 += X[q] 
    end
    return Symmetric( 2.  * [s11  s12 ; s12 s22 ])
end


resder (generic function with 1 method)

In [6]:
sumres(param)

3.632498529617096e9

In [7]:
dersumres(param)

2-element Vector{Float64}:
 3.6252495786463314e8
 1.8108028521374087e6

In [8]:
resder(param)

LoadError: UndefVarError: `Symmetric` not defined

On utilise un Levenberg Marquardt construit comme suit :
Il y a toujours deux cas selon que nabla r nabla r^top est inversible.

Le cas qui fait planter l'algo est le cas nablar nablar top = 0

- Si elle est inversible, on inverse et on tente la direction
Si la direction n'est pas bonne on initialise le pas de descente de gradient avec l'inverse de la plus grande valeur propre de nabla r nabla r^top (un peu à la Baillon Haddad : voir norme de Lipschitz du gradient et coericvité)

- Si elle n'est pas inversible, idem.

Dans tous les cas à chaque itération on se laisse la possibilité de ne faire que du gauss newton.

Si on fait de la recherche de pas on a résoudre (A + mu I) x = f
Cela vaut le coup d'utiliser la décomposition de Hessenberg de A qui permet de bien faire ce genre de choses. C'est aussi facile davoir accès au det à moindre coup une fois cette décomposition calculer. Même le calcul des valeurs est simplifiés car si A = Q H Q^\top alors H et A ont les mêmes valeurs propres et H est tridiagonale donc ces valeurs propres sont plus faciles à calculer.

In [294]:
function distance(x::Vector{Float64})::Float64
    return sqrt(x[1]^2 + x[2]^2)
end

LMdescent(sumres, dersumres, resder, param, distance)

2-element Vector{Float64}:
 -5.000028582084913
 11.999548041431193

In [17]:
include("Optimisation.jl")
using LinearAlgebra
Ndata = 300
X = [i for i in 1:Ndata]
F = 2. * sin.(3. .* X) .+ 3. * cos.(2. .* X) 
param = [2. ; 3.]

function sumres(param::Vector{Float64})::Float64
    s = 0.
    for q in 1:Ndata
        s+= ( param[1] * sin(param[2] * X[q]) + param[2] * cos(param[1] * X[q])  - F[q])^2 
    end
    return s 
end

function dersumres(param::Vector{Float64})::Vector{Float64}
    # renvoie le gradient de l function coût
    m = size(param)[1]
    s1 = 0.
    s2 = 0.
    for q in 1:Ndata
        s1+=  (sin(param[2] * X[q]) - param[2] * X[q] * sin(param[1] * X[q]))* ( param[1] * sin(param[2] * X[q]) + param[2] * cos(param[1] * X[q])  - F[q])
        s2 +=  ( param[1] * X[q] * cos(param[2] * X[q]) +  cos(param[1] * X[q]))* ( param[1] * sin(param[2] * X[q]) + param[2] * cos(param[1] * X[q])  - F[q])
    end
    return 2. * [s1  ; s2 ]
end

function resder(param::Vector{Float64})::Symmetric{Float64, Matrix{Float64}}
    # doit renvoyer une matrice symétrique de taille 2 * 2 ici
    m = size(param)[1]
    s11 = 0.
    s12 = 0.
    s22 = 0.
    for q in 1:Ndata
        s11 +=  ( sin(param[2] * X[q]) - param[2] * X[q] * sin(param[1] * X[q]) )^2
        s22 += ( sin(param[2] * X[q]) - param[2] * X[q] * sin(param[1] * X[q]) ) * ( param[1] * X[q] * cos(param[2] * X[q]) +  X[q] * cos(param[1] * X[q]))
        s12 += ( param[1] * X[q] * cos(param[2] * X[q]) +  cos(param[1] * X[q]))^2
    end
    return Symmetric( 2.  * [s11  s12 ; s12 s22 ])
end

function distance(x::Vector{Float64})::Float64
    return sqrt(x[1]^2 + x[2]^2)
end

LMdescent(sumres, dersumres, resder, param, distance)

avant
5852.942013310626
apres
5793.31138049353
avant
5793.31138049353
apres
5760.178749613451
avant
5760.178749613451
apres
5856.3355175760325
 Number of iterations for descent research exceeded !
The value of the cost before stopping is : 5856.3355175760325


2-element Vector{Float64}:
 3.0001621225494417
 3.9857856999519883